In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, count, when
from datetime import datetime

# Read the silver table for quality checking
silver_df = spark.read.table("workspace.default.silver_transactions")

# Initialize a list to hold audit results
audit_results = []

# --- Check 1: Completeness - null transaction_ids ---
null_txn_count = silver_df.filter(col("transaction_id").isNull()).count()
audit_results.append((
    "completeness_transaction_id",
    "pass" if null_txn_count == 0 else "fail",
    null_txn_count,
    datetime.now()
))

# --- Check 2: Completeness - null customer_ids ---
null_cust_count = silver_df.filter(col("customer_id").isNull()).count()
audit_results.append((
    "completeness_customer_id",
    "pass" if null_cust_count == 0 else "fail",
    null_cust_count,
    datetime.now()
))

# --- Check 3: Value range - amount must be positive and under 100000 ---
out_of_range_count = silver_df.filter(
    (col("amount") <= 0) | (col("amount") > 100000)
).count()
audit_results.append((
    "range_amount_valid",
    "pass" if out_of_range_count == 0 else "fail",
    out_of_range_count,
    datetime.now()
))

# --- Check 4: Referential pattern - currency must be a known code ---
valid_currencies = ["USD", "EUR", "GBP", "JPY", "CAD"]
invalid_currency_count = silver_df.filter(
    ~col("currency").isin(valid_currencies)
).count()
audit_results.append((
    "referential_currency_valid",
    "pass" if invalid_currency_count == 0 else "fail",
    invalid_currency_count,
    datetime.now()
))

# --- Check 5: Uniqueness - no duplicate transaction_ids ---
total_rows = silver_df.count()
distinct_rows = silver_df.select("transaction_id").distinct().count()
duplicate_count = total_rows - distinct_rows
audit_results.append((
    "uniqueness_transaction_id",
    "pass" if duplicate_count == 0 else "fail",
    duplicate_count,
    datetime.now()
))

print(f"Quality checks complete. {len(audit_results)} checks ran.")

In [0]:
display(audit_results)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Define the schema for audit results
audit_schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("check_result", StringType(), False),
    StructField("records_flagged", IntegerType(), False),
    StructField("run_timestamp", TimestampType(), False)
])

# Create a DataFrame from the audit results
audit_df = spark.createDataFrame(audit_results, schema=audit_schema)

# Write to the quality_audit Delta table in append mode
audit_df.write.mode("append").saveAsTable("workspace.default.quality_audit")

print("Audit results written to workspace.default.quality_audit")

In [0]:
display(audit_df)

In [0]:
# Display the audit history sorted by most recent run
result_df = spark.sql("""
    SELECT check_name, check_result, records_flagged, run_timestamp
    FROM workspace.default.quality_audit
    ORDER BY run_timestamp DESC, check_name
""")

result_df.display()